# Практическая работа №9. Метод опорных векторов (SVM)

**Цель:** исследовать влияние ядра и гиперпараметров `C` и `gamma` на качество классификатора. Работа выполнена на датасете диагностики рака молочной железы.

Масштабирование включено в `Pipeline`, поэтому при кросс-валидации статистики стандартизации вычисляются только по обучающей части каждого фолда. Тестовая выборка не участвует в выборе гиперпараметров.

## 1. Загрузка данных и фиксированное разбиение

In [ ]:
from threadpoolctl import threadpool_limits
threadpool_limits(1)  # стабильная производительность в средах с ограниченными ресурсами
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
import time

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
data = pd.read_csv(ROOT / 'data' / 'classification_breast_cancer.csv')
X = data.drop(columns=['target', 'target_name'])
y = data['target'].astype(int)  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('Размер данных:', X.shape, '| признаков:', X.shape[1])
print('train/test:', X_train.shape, X_test.shape)
print('Доли классов в train:', y_train.value_counts(normalize=True).sort_index().round(3).to_dict())

## 2. Grid Search: ядро, C и gamma

Сравним линейное и RBF-ядро по macro F1 на стратифицированной пятиблочной кросс-валидации. Шаги масштабирования и классификации находятся в одном пайплайне.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline([('scale', StandardScaler()), ('svc', SVC())])
param_grid = [
    {'svc__kernel': ['linear'], 'svc__C': [0.1, 1, 10, 100]},
    {'svc__kernel': ['rbf'], 'svc__C': [0.1, 1, 10, 100],
     'svc__gamma': ['scale', 0.001, 0.01, 0.1, 1]},
]
search = GridSearchCV(pipe, param_grid, scoring='f1_macro', cv=cv, n_jobs=1,
                      return_train_score=True)
t0 = time.perf_counter()
search.fit(X_train, y_train)
search_seconds = time.perf_counter() - t0
best_model = search.best_estimator_
pred = best_model.predict(X_test)
print('Лучшие параметры:', search.best_params_)
print('Лучшая CV macro F1:', round(search.best_score_, 4))
print('Test accuracy:', round(accuracy_score(y_test, pred), 4))
print('Test macro F1:', round(f1_score(y_test, pred, average='macro'), 4))
print('Время подбора (сек):', round(search_seconds, 2))
print(classification_report(y_test, pred, target_names=['malignant', 'benign']))

## 3. Тепловая карта качества для RBF

Покажем средний macro F1 по CV в зависимости от `C` и `gamma`. `gamma='scale'` показана отдельно текстом в таблице результатов поиска, поскольку это не фиксированное числовое значение.

In [ ]:
# Таблица и тепловая карта для числовых gamma
results = pd.DataFrame(search.cv_results_)
rbf = results[results['param_svc__kernel'] == 'rbf'].copy()
rbf['C'] = rbf['param_svc__C'].astype(float)
rbf['gamma'] = rbf['param_svc__gamma'].astype(str)
heat = rbf[rbf['gamma'] != 'scale'].pivot(index='C', columns='gamma', values='mean_test_score')
display(heat.round(4))
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(heat.values, aspect='auto', cmap='viridis', vmin=heat.values.min(), vmax=heat.values.max())
ax.set_xticks(range(len(heat.columns)), labels=heat.columns)
ax.set_yticks(range(len(heat.index)), labels=heat.index)
ax.set_xlabel('gamma'); ax.set_ylabel('C'); ax.set_title('RBF: средний CV macro F1')
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        ax.text(j, i, f'{heat.values[i, j]:.3f}', ha='center', va='center', color='white' if heat.values[i,j] < (heat.values.min()+heat.values.max())/2 else 'black')
fig.colorbar(im, ax=ax, label='CV macro F1'); plt.tight_layout(); plt.show()

## 4. Влияние C, сравнение с линейным SVM и baseline

`C` управляет штрафом за ошибки: малое значение допускает больше ошибок ради широкой и простой границы; большое сильнее подгоняет модель под обучающие данные. Ниже сравним train и CV для RBF. Дополнительно проверим качество линейного SVM и простого baseline на том же разбиении.

In [ ]:
cs = [0.01, 0.1, 1, 10, 100]
train_scores, cv_scores = [], []
for c in cs:
    model = Pipeline([('scale', StandardScaler()), ('svc', SVC(C=c, kernel='rbf', gamma='scale'))])
    model.fit(X_train, y_train)
    train_scores.append(f1_score(y_train, model.predict(X_train), average='macro'))
    cv_scores.append(cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=1).mean())
plt.figure(figsize=(7, 4))
plt.semilogx(cs, train_scores, marker='o', label='train')
plt.semilogx(cs, cv_scores, marker='o', label='CV')
plt.xlabel('C'); plt.ylabel('macro F1'); plt.title('Влияние C для RBF SVM'); plt.legend(); plt.grid(True); plt.show()

linear = Pipeline([('scale', StandardScaler()), ('svc', SVC(kernel='linear', C=1))])
t0 = time.perf_counter(); linear.fit(X_train, y_train); linear_seconds = time.perf_counter()-t0
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
models = [('Dummy baseline', baseline, 0.0), ('Linear SVM (C=1)', linear, linear_seconds),
          ('Tuned SVM', best_model, search_seconds)]
comparison = []
for name, model, seconds in models:
    comparison.append({'model': name,
        'train_macro_F1': f1_score(y_train, model.predict(X_train), average='macro'),
        'test_macro_F1': f1_score(y_test, model.predict(X_test), average='macro'),
        'test_accuracy': accuracy_score(y_test, model.predict(X_test)),
        'fit_seconds': seconds})
display(pd.DataFrame(comparison).round(4))

## 5. Выводы и ответы на вопросы

1. Выбранное ядро и параметры указаны в результатах Grid Search; качество выбора оценивалось только по CV на train.
2. Сравните CV результаты линейного и RBF вариантов в `search.cv_results_`. RBF позволяет моделировать нелинейную границу; линейное ядро ограничено гиперплоскостью.
3. По графику train/CV видно, что чрезмерно большое `C` может повышать качество на train без сопоставимого выигрыша на CV — признак переобучения. Малое `C` задаёт более мягкий зазор. `gamma` определяет локальность влияния объектов: слишком большое значение создаёт сложную границу и повышает риск переобучения.
4. Таблица сравнения содержит train и test macro F1 и время обучения. Test использован только для финальной оценки после подбора.

**Ограничения:** результат зависит от конкретного разбиения и набора проверенных параметров. Сравнение с ПР-7/8 можно считать честным только при использовании тех же данных, разбиения и метрики; результаты этих работ в текущем проекте не подгружаются автоматически. У SVM обязательное масштабирование признаков.